In [ ]:
%pip install -q -U langchain langchain-openai langgraph pydantic

import os
from getpass import getpass
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Ask for the API key only if it is not already set
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Sample data (approximate specs)
VEHICLES = [
    {"name": "Koenigsegg Jesko",     "type": "car",  "power_hp": 1600, "top_speed_kmh": 480},
    {"name": "Bugatti Chiron",       "type": "car",  "power_hp": 1578, "top_speed_kmh": 440},
    {"name": "Ferrari SF90",         "type": "car",  "power_hp": 986,  "top_speed_kmh": 340},
    {"name": "Kawasaki Ninja H2R",   "type": "bike", "power_hp": 326,  "top_speed_kmh": 400},
    {"name": "Ducati Panigale V4 R", "type": "bike", "power_hp": 240,  "top_speed_kmh": 299},
    {"name": "Suzuki Hayabusa",      "type": "bike", "power_hp": 190,  "top_speed_kmh": 299},
]

# ---------------------------------------------------------------
# 1) RULE GENERATOR (uses AI)
#    The AI turns a sentence into a structured rule.
# ---------------------------------------------------------------
class Rule(BaseModel):
    label: str = Field(description="Short rule name, e.g. 'fast bike'")
    vehicle_type: Literal["car", "bike", "any"]
    field: Literal["power_hp", "top_speed_kmh"]
    operator: Literal[">", "<"]
    value: float

rule_writer = model.with_structured_output(Rule)
RULES = {}   # saved rules: label -> Rule

@tool
def create_rule(sentence: str) -> str:
    """Turn a plain English sentence into a rule and save it.
    Use this when the user defines a new rule, for example 'a fast bike has top speed over 300'."""
    rule = rule_writer.invoke(f"Convert this sentence into a rule: {sentence}")
    RULES[rule.label.lower()] = rule
    return f"Rule saved: '{rule.label}' = {rule.vehicle_type} with {rule.field} {rule.operator} {rule.value}"

# ---------------------------------------------------------------
# 2) RULE ENGINE (plain code, no AI)
#    Same rule + same data = same result, every time.
# ---------------------------------------------------------------
def passes(rule: Rule, vehicle: dict) -> bool:
    if rule.vehicle_type != "any" and vehicle["type"] != rule.vehicle_type:
        return False
    value = vehicle[rule.field]
    return value > rule.value if rule.operator == ">" else value < rule.value

@tool
def check_vehicles(rule_label: str) -> str:
    """Apply a saved rule to every vehicle and report which ones pass. Call create_rule first if the rule does not exist."""
    rule = RULES.get(rule_label.lower())
    if rule is None:
        return f"No rule named '{rule_label}'. Saved rules: {list(RULES) or 'none'}"
    lines = []
    for v in VEHICLES:
        result = "PASS" if passes(rule, v) else "fail"
        lines.append(f"{v['name']} ({v['type']}, {v[rule.field]}): {result}")
    return "\n".join(lines)

# ---------------------------------------------------------------
# 3) AI AGENT (uses AI)
#    Understands the request, calls the tools, explains the result.
# ---------------------------------------------------------------
agent = create_agent(
    model=model,
    tools=[create_rule, check_vehicles],
    system_prompt=(
        "You help users define rules and check vehicles against them. "
        "If the user defines a new rule, call create_rule, then check_vehicles. "
        "Never decide pass or fail yourself; use only check_vehicles results. "
        "Explain the result in one or two simple sentences."
    ),
)

# Label which tool is AI and which is plain code, so you can see the difference
KIND = {"create_rule": "AI rule generator", "check_vehicles": "CODE rule engine"}

# Try: A fast bike has top speed over 300 km/h
# Try: A powerful car has more than 1000 hp
while True:
    text = input("\nRequest (or 'quit'): ").strip()
    if text.lower() == "quit":
        break
    if text:
        result = agent.invoke({"messages": [{"role": "user", "content": text}]})
        for m in result["messages"]:
            for call in getattr(m, "tool_calls", []):
                print(f"[{KIND[call['name']]}] {call['name']}({call['args']})")
            if m.type == "tool":
                print(m.content)
        print("Answer:", result["messages"][-1].content)